In [17]:
"""
This file completes the estimation of county-year business formations to produce the base of the panel for analysis.
"""

# #################################
# SETUP

# Packages
import pandas as pd

# Loading Datasets
state_forms = pd.read_csv('../data_clean/annualized_state_formations.csv',
                           dtype={'full_fips': str, 'state_fips': str, 'county_fips': str})
state_apps = pd.read_csv('../data_clean/state_apps_annual.csv',
                         dtype={'full_fips': str, 'state_fips': str, 'county_fips': str})
county_apps = pd.read_csv('../data_clean/annual_county_applications.csv',
                          dtype={'full_fips': str, 'state_fips': str, 'county_fips': str})

## OG Missing
Getting dataframes of missing observations in all three datasets used in the calculation to compare with the missing observations in the resulting merged estimation dataset to determine true missing vs merge issue.

In [18]:
# State Formations
sf_og_missing = state_forms[state_forms['state_formations'].isna()]
sf_og_missing.to_csv('sf_og_missing.csv')

# State Applications
sa_og_missing = state_apps[state_apps['state_apps'].isna()]
sa_og_missing.to_csv('sa_og_missing.csv')

# County Applications
ca_og_missing = county_apps[county_apps['county_apps'].isna()]
ca_og_missing.to_csv('ca_og_missing.csv')

## Updated county_formations_estimation.py file
All actual code edits happen in the .py file. Copy-paste updated file AFTER setup section here and comment out final line. 

In [19]:

# #################################
# MERGING DATASETS

# Merging state formations and state applications
state = pd.merge(state_forms, state_apps, on=['year', 'state_fips', 'STATE', 'STATE_NAME'])
assert state.groupby(['state_fips', 'year']).size().eq(1).all()

# Merging state dataframe with county applications
est = pd.merge(county_apps, state, on=['year', 'state_fips', 'STATE'], validate='many_to_one')
assert est.groupby(['full_fips', 'year']).size().eq(1).all()

# Merge validation
assert est['state_apps'].notna().all()

# Columns Cleanup
est = est.rename(columns={'County': 'county'})
est = est.reindex(columns = ['STATE', 'year', 'county', 'full_fips', 'state_formations', 'state_apps', 'county_apps', 'state_fips', 'county_fips', 'STATE_NAME'])


# #################################
# ESTIMATING COUNTY FORMATIONS

# Correcting variable types
est['county_apps'] = pd.to_numeric(est['county_apps'], errors='coerce')

# Calculating estimated county formations
est['county_forms_est'] = (est['state_formations']/est['state_apps']) * est['county_apps']


# #################################
# SAVING DATASET
# est.to_csv('data_clean/county_formations_estimation.csv', index=False)

## Investigation Code:
Debugging shenanigans start here

In [20]:
# FINDING MISSINGS!
sf_missing = est[est['state_formations'].isna()]
sa_missing = est[est['state_apps'].isna()]
ca_missing = est[est['county_apps'].isna()]

In [21]:
# Merging sf missing dataframes to separate out true missing (47) vs the large remainder of missings in the final data
sf_WHAT = pd.merge(sf_missing, sf_og_missing, on=['state_fips', 'year'], indicator=True)

sf_missing_problematic = sf_WHAT[sf_WHAT['_merge']!= 'both']
